<a href="https://colab.research.google.com/github/maneesha-bogahawatta/J26-DS-320/blob/member4%2Fcritical-incidents/%20%20components/c4_critical_incidents/fall_branch/C4_fall_detection_wild_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C4 Fall detection: real-world ("wild") test on social-media clips

Tests **your existing pipeline** (MediaPipe 6 features → 100-step sequence → CNN-LSTM) on the 11 social-media clips,
**without training on them**. This is the staged → wild check from your proposal (Sec 3.5).

**Before running:** upload the 11 `snaptik_..._v3.mp4` files to Drive at
`MyDrive/J26-DS-320/datasets/social_media_wild/`.

Run the cells in order. Cell 3 (pose extraction) saves progress, so a runtime reset doesn't lose work.

In [ ]:
# Cell 1: setup (Colab)
from google.colab import drive
drive.mount('/content/drive')

!pip -q install mediapipe
!wget -q -O pose_landmarker_lite.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task

In [ ]:
# Cell 2: paths, imports and ground truth
import os, re, cv2, json, copy, pickle, time
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

BASE = '/content/drive/MyDrive/J26-DS-320/datasets'
WILD = f'{BASE}/social_media_wild'            # the 11 mp4 files
FEAT = f'{BASE}/wild_features.pkl'            # pose features (Cell 3 output)
OUT  = f'{BASE}/wild_test_results.json'       # final results (Cell 7 output)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Ground truth in seconds of the ORIGINAL video (read from frames at 2 fps, so +/-0.5 s; verify in CVAT).
#   skip_before : ignore the first N seconds (reaction-video intro / monitor bezel)
#   fall        : start = descent begins, end = lands, down_until = starts getting up (or clip ends)
#   negative    : bystanders bending / kneeling / crouching -> an alert here is a FALSE ALARM
#   domain      : 'wild' = real accident, 'staged' = acted prank, 'exclude' = ambiguous
GT = {
 '6952980706943405317': dict(cam='cam01', domain='wild', skip_before=2.5, view='cctv_refilmed',
     falls=[(6.5, 8.5, 10.5)], negatives=[]),
 '7160510248325745947': dict(cam='cam02', domain='wild', skip_before=0, view='handheld',
     falls=[(1.0, 3.0, 10.0)], negatives=[]),
 '6805425229368806661': dict(cam='cam03', domain='wild', skip_before=0, view='handheld',
     falls=[(2.5, 4.0, 12.0)], negatives=[(12.5, 30.0)]),
 '6770167785269120261': dict(cam='cam04', domain='wild', skip_before=0, view='handheld_overhead',
     falls=[(3.5, 5.0, 14.0)], negatives=[(14.5, 29.0)]),
 '7522158970664963342': dict(cam='cam05', domain='wild', skip_before=2.5, view='cctv_refilmed',
     falls=[(5.5, 7.5, 8.5)], negatives=[]),
 '7347119749287005482': dict(cam='cam06', domain='wild', skip_before=0, view='handheld',
     falls=[(2.9, 4.3, 5.7)], negatives=[]),
 '7603843380648398102': dict(cam='cam07', domain='wild', skip_before=0, view='cctv',
     falls=[(2.5, 3.5, 6.0)], negatives=[]),
 '7637913967943421204': dict(cam='cam08', domain='exclude', skip_before=0, view='cctv',
     falls=[(3.5, 4.5, 7.0)], negatives=[]),          # child tumbling with a cart - ambiguous
 '7633868912903441682': dict(cam='cam09', domain='wild', skip_before=0, view='cctv',
     falls=[(14.0, 16.0, 38.0)], negatives=[]),
 '7660902506318417165': dict(cam='cam10', domain='wild', skip_before=0, view='cctv',
     falls=[(3.5, 4.5, 13.2)], negatives=[(6.5, 13.2)]),
 '7173323503750434053': dict(cam='cam11', domain='staged', skip_before=0, view='handheld',
     falls=[(0.0, 1.0, 20.0), (33.0, 34.0, 40.0)], negatives=[]),
}

files = {re.search(r'snaptik_(\d+)_v3', f).group(1): os.path.join(WILD, f)
         for f in os.listdir(WILD) if f.endswith('.mp4') and 'snaptik_' in f and not f.startswith('.')}
missing = [k for k in GT if k not in files]
print('videos found:', len(files), '| missing:', missing or 'none')

In [ ]:
# Cell 3: pose features - SAME 6 features as your URFD / GMDCSA notebooks
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

detector = vision.PoseLandmarker.create_from_options(vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path='pose_landmarker_lite.task'),
    output_segmentation_masks=False))

def frame_features(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res = detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb))
    if not res.pose_landmarks: return None
    lm = res.pose_landmarks[0]
    xs, ys = [l.x for l in lm], [l.y for l in lm]
    w, h = max(xs)-min(xs), max(ys)-min(ys)
    return {"nose_y": lm[0].y,
            "shoulder_y": (lm[11].y + lm[12].y)/2,
            "hip_y": (lm[23].y + lm[24].y)/2,
            "aspect_ratio": w/h if h > 0.01 else 0,
            "width": w, "height": h}

In [ ]:
# Cell 4: extract features for every clip (every 2nd frame, with timestamps; saves progress)
STRIDE = 2
feats_all = pickle.load(open(FEAT, 'rb')) if os.path.exists(FEAT) else {}
t0 = time.time()
for vid, g in GT.items():
    if vid in feats_all or vid not in files: continue
    cap = cv2.VideoCapture(files[vid])
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    feats, i = [], 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        t = i / fps
        if i % STRIDE == 0 and t >= g['skip_before']:
            f = frame_features(frame)
            if f: f['t'] = t
            feats.append(f if f else {'t': t, 'none': True})
        i += 1
    cap.release()
    feats_all[vid] = {'fps': fps, 'duration': i / fps, 'features': feats}
    pickle.dump(feats_all, open(FEAT, 'wb'))
    print(f"{g['cam']} done | {len(feats)} frames | {time.time()-t0:.0f}s")

# sanity check: how often does MediaPipe find a person, overall and DURING the fall?
print(f"\n{'cam':6} {'view':18} {'pose found':>10} {'during fall':>12}")
for vid, g in GT.items():
    fr = feats_all[vid]['features']
    ok = [f for f in fr if 'none' not in f]
    s, _, d = g['falls'][0]
    inwin = [f for f in fr if s <= f['t'] <= d]
    okwin = [f for f in inwin if 'none' not in f]
    print(f"{g['cam']:6} {g['view']:18} {len(ok)/max(1,len(fr)):10.0%} {len(okwin)/max(1,len(inwin)):12.0%}")

In [ ]:
# Cell 5: model - your FallCNNLSTM, trained on ALL 160 GMDCSA-24 clips (your best setup: 6 features)
# Trains 3 seeds once and saves them; later runs just load them.
class FallCNNLSTM(nn.Module):
    def __init__(self, num_features=6, cnn_channels=[32, 64, 128], lstm_hidden=128, lstm_layers=2):
        super().__init__()
        self.conv1 = nn.Conv1d(num_features, cnn_channels[0], kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(cnn_channels[0], cnn_channels[1], kernel_size=3, padding=1)
        self.conv3 = nn.Conv1d(cnn_channels[1], cnn_channels[2], kernel_size=3, padding=1)
        self.relu = nn.ReLU(); self.dropout = nn.Dropout(0.3)
        self.lstm = nn.LSTM(cnn_channels[2], lstm_hidden, num_layers=lstm_layers, batch_first=True, dropout=0.3)
        self.fc1 = nn.Linear(lstm_hidden, 64); self.fc2 = nn.Linear(64, 2)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.dropout(self.relu(self.conv1(x)))
        x = self.dropout(self.relu(self.conv2(x)))
        x = self.dropout(self.relu(self.conv3(x)))
        _, (h_n, _) = self.lstm(x.permute(0, 2, 1))
        return self.fc2(self.relu(self.fc1(h_n[-1])))

def loader(Xa, ya, shuffle):
    return DataLoader(TensorDataset(torch.tensor(Xa), torch.tensor(ya, dtype=torch.long)), batch_size=8, shuffle=shuffle)

def predict(model, Xa):
    model.eval()
    with torch.no_grad():
        return torch.argmax(model(torch.tensor(Xa).to(device)), dim=1).cpu().numpy()

def predict_proba(model, Xa):
    model.eval()
    with torch.no_grad():
        return torch.softmax(model(torch.tensor(Xa).to(device)), dim=1)[:, 1].cpu().numpy()

def train_model(Xtr, ytr, Xva, yva, epochs=50, patience=10):
    model = FallCNNLSTM().to(device)
    w = torch.tensor([1.0 / c for c in np.bincount(ytr)], dtype=torch.float32).to(device)
    crit = nn.CrossEntropyLoss(weight=w)
    opt = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
    best_f1, best_state, bad = -1, None, 0
    for ep in range(epochs):
        model.train()
        for bx, by in loader(Xtr, ytr, True):
            bx, by = bx.to(device), by.to(device)
            opt.zero_grad(); crit(model(bx), by).backward(); opt.step()
        f1 = f1_score(yva, predict(model, Xva), zero_division=0)
        if f1 > best_f1: best_f1, best_state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    return model

d = pickle.load(open(f'{BASE}/gmdcsa24_arrays.pkl', 'rb'))
X, y = d['X'].astype(np.float32), np.array(d['y'])
models = {}
ens = []
for seed in [0, 1, 2]:
    path = f'{BASE}/fall_cnn_lstm_gmdcsa_final_s{seed}.pt'
    m = FallCNNLSTM().to(device)
    if os.path.exists(path):
        m.load_state_dict(torch.load(path, map_location=device))
    else:
        torch.manual_seed(seed); np.random.seed(seed)
        Xtr, Xva, ytr, yva = train_test_split(X, y, test_size=0.15, stratify=y, random_state=seed)
        m = train_model(Xtr, ytr, Xva, yva)
        torch.save(m.state_dict(), path)
    ens.append(m)
    print('seed', seed, 'ready')
models['gmdcsa_6f_ensemble'] = ens

# your combined URFD+GMDCSA model, for comparison
v2 = f'{BASE}/fall_cnn_lstm_v2.pt'
if os.path.exists(v2):
    m = FallCNNLSTM().to(device); m.load_state_dict(torch.load(v2, map_location=device))
    models['combined_v2'] = [m]
print('models:', list(models))

In [ ]:
# Cell 6: sliding windows over each clip -> fall probability per window
# Your model was trained on whole 5-8 s clips resampled to 100 steps, so we slide a
# 6 s window in 1 s steps and resample each window the same way (by time).
FEATURE_NAMES = ["nose_y", "shoulder_y", "hip_y", "aspect_ratio", "width", "height"]
WIN, STEP, T = 6.0, 1.0, 100

def window_to_array(feats, t0, t1):
    valid = [f for f in feats if 'none' not in f and t0 <= f['t'] <= t1]
    if len(valid) < 5: return None                      # not enough pose -> no decision
    t = np.array([f['t'] for f in valid])
    grid = np.linspace(t.min(), t.max(), T)
    return np.stack([np.interp(grid, t, [f[n] for f in valid]) for n in FEATURE_NAMES], axis=1).astype(np.float32)

windows = {}   # vid -> list of dicts (t0, t1, probs per model)
for vid, g in GT.items():
    fr = feats_all[vid]['features']
    start, end = g['skip_before'], feats_all[vid]['duration']
    starts = np.arange(start, max(start, end - WIN) + 1e-6, STEP) if end - start > WIN else [start]
    rows = []
    for t0 in starts:
        t1 = min(t0 + WIN, end)
        arr = window_to_array(fr, t0, t1)
        p = {name: (float(np.mean([predict_proba(m, arr[None])[0] for m in ms])) if arr is not None else None)
             for name, ms in models.items()}
        rows.append({'t0': round(float(t0), 2), 't1': round(float(t1), 2), 'p': p})
    windows[vid] = rows
    print(g['cam'], len(rows), 'windows | no-pose windows:', sum(r['p'][list(models)[0]] is None for r in rows))

In [ ]:
# Cell 7: event-level evaluation (same rules as the kit's evaluate.py)
# An alert = a window with p >= THR; consecutive alert windows merge into one alert episode.
# Alert time = END of the first alert window (that's when the system could actually raise it).
# Fall detected  : an alert episode starts inside [fall.start - 0.5, down_until + 2]
# False alarm    : an alert episode outside every fall window (on a negative = fall-like false alarm)
GRACE, PRE = 2.0, 0.5

def episodes(rows, name, thr):
    eps, cur = [], None
    for r in rows:
        on = r['p'][name] is not None and r['p'][name] >= thr
        if on and cur is None: cur = {'t_alert': r['t1'], 't0': r['t0'], 'pmax': r['p'][name]}
        elif on: cur['pmax'] = max(cur['pmax'], r['p'][name])
        elif cur is not None: eps.append(cur); cur = None
    if cur: eps.append(cur)
    return eps

def evaluate(name, thr, domains=('wild', 'staged')):
    falls = detected = fa = fa_neg = 0; lat = []; per = []
    for vid, g in GT.items():
        if g['domain'] not in domains: continue
        eps = episodes(windows[vid], name, thr)
        hit = [False] * len(g['falls'])
        for e in eps:
            # does this episode overlap a fall window?
            m = [k for k, (s, l, du) in enumerate(g['falls']) if e['t0'] <= du + GRACE and e['t_alert'] >= s - PRE]
            if m:
                k = m[0]
                if not hit[k]: hit[k] = True; lat.append(max(0.0, e['t_alert'] - g['falls'][k][0]))
            else:
                fa += 1
                if any(e['t0'] <= ne and e['t_alert'] >= ns for ns, ne in g['negatives']): fa_neg += 1
        falls += len(hit); detected += sum(hit)
        per.append({'cam': g['cam'], 'domain': g['domain'], 'falls': len(hit), 'detected': sum(hit),
                    'alerts': len(eps), 'max_p_in_fall': max([r['p'][name] or 0 for r in windows[vid]
                        if any(r['t0'] <= du and r['t1'] >= s for s, l, du in g['falls'])], default=0)})
    hours = sum(feats_all[v]['duration'] for v, g in GT.items() if g['domain'] in domains) / 3600
    tp_eps = detected  # one credited episode per detected fall
    prec = tp_eps / (tp_eps + fa) if tp_eps + fa else 0
    rec = detected / falls if falls else 0
    return {'model': name, 'thr': thr, 'domains': list(domains), 'falls': falls, 'detected': detected,
            'recall': round(rec, 3), 'precision': round(prec, 3),
            'f1': round(2*prec*rec/(prec+rec), 3) if prec + rec else 0.0,
            'false_alarms': fa, 'false_alarms_on_fall_like': fa_neg,
            'false_alarms_per_hour': round(fa / hours, 1) if hours else None,
            'latency_mean_s': round(float(np.mean(lat)), 2) if lat else None}, per

results = {}
for name in models:
    for thr in [0.40, 0.50]:
        for doms, tag in [(('wild',), 'wild'), (('staged',), 'staged'), (('wild', 'staged'), 'all')]:
            s, per = evaluate(name, thr, doms)
            results[f'{name}@{thr}/{tag}'] = {'summary': s, 'per_clip': per}
            print(f"{name:20} thr={thr:.2f} {tag:6} recall={s['recall']:.2f} ({s['detected']}/{s['falls']})  "
                  f"precision={s['precision']:.2f}  F1={s['f1']:.2f}  FA={s['false_alarms']} "
                  f"(fall-like {s['false_alarms_on_fall_like']})  latency={s['latency_mean_s']}")

print('\nPer clip, best model @0.40:')
for r in results[f'{list(models)[0]}@0.4/all']['per_clip']:
    print(r)

json.dump({'window_s': WIN, 'step_s': STEP, 'results': results}, open(OUT, 'w'), indent=2)
print('\nsaved ->', OUT)

### How to read Cell 7

* **Recall on `wild`** is the headline number: how many real accidental falls your GMDCSA-trained model catches.
  Compare it with your lab result (LOSO F1 0.90) to show the **staged → wild gap**.
* **Do not trust precision / false alarms from these clips.** Most clips are only 6-15 s long, so almost every
  6 s window overlaps the fall. False alarms need long *normal* footage (CAVIAR normal clips, UCF-Crime normal videos).
  Report false alarms from those, not from here.
* **Check the `pose found during fall` column from Cell 4 first.** If MediaPipe finds nobody while the person is on the floor,
  a miss is a perception failure, not a model failure. Say which one it is in your write-up.
* MediaPipe Pose tracks **one person per frame**. In crowded clips (cam02, cam03, cam04) it may follow a bystander
  instead of the person who fell. That's a real limitation to report, and a reason to move to a multi-person pose base.

In [ ]:
# Cell 8: figure for your report - fall probability over time for each clip (fall shaded red)
import matplotlib.pyplot as plt
name = list(models)[0]
vids = [v for v in GT if GT[v]['domain'] != 'exclude']
fig, axes = plt.subplots(len(vids), 1, figsize=(10, 1.6 * len(vids)), sharey=True)
for ax, vid in zip(axes, vids):
    g = GT[vid]
    xs = [(r['t0'] + r['t1']) / 2 for r in windows[vid]]
    ps = [r['p'][name] if r['p'][name] is not None else np.nan for r in windows[vid]]
    for s, l, du in g['falls']: ax.axvspan(s, du, color='red', alpha=0.15)
    for ns, ne in g['negatives']: ax.axvspan(ns, ne, color='orange', alpha=0.15)
    ax.plot(xs, ps, marker='o', ms=3); ax.axhline(0.40, ls='--', c='gray', lw=0.8)
    ax.set_ylim(0, 1); ax.set_ylabel(g['cam'], rotation=0, labelpad=20)
axes[-1].set_xlabel('time (s)  |  red = fall, orange = fall-like negative, dashed = threshold 0.40')
plt.tight_layout(); plt.savefig(f'{BASE}/wild_test_timeline.png', dpi=130); plt.show()

In [ ]:
# Cell 7b: alert only after k consecutive windows above the threshold (Fig 4.3 'persistent fallen state')
def episodes_k(rows, name, thr, k=2):
    eps, run, cur = [], 0, None
    for r in rows:
        on = r['p'][name] is not None and r['p'][name] >= thr
        run = run + 1 if on else 0
        if run == k and cur is None:
            cur = {'t_alert': r['t1'], 't0': r['t0'], 'pmax': r['p'][name]}
        elif not on and cur is not None:
            eps.append(cur); cur = None
    if cur: eps.append(cur)
    return eps

_orig_episodes = episodes
persist = {}
for k in [1, 2, 3]:
    episodes = lambda rows, name, thr, k=k: episodes_k(rows, name, thr, k)
    for thr in [0.40, 0.50]:
        for tag, doms in [('wild', ('wild',)), ('staged', ('staged',))]:
            s, _ = evaluate('gmdcsa_6f_ensemble', thr, doms)
            persist[f'k{k}@{thr}/{tag}'] = s
            print(f"k={k} thr={thr:.2f} {tag:6} recall={s['recall']:.2f} ({s['detected']}/{s['falls']})  "
                  f"FA={s['false_alarms']} (fall-like {s['false_alarms_on_fall_like']})  latency={s['latency_mean_s']}")
episodes = _orig_episodes
json.dump(persist, open(f'{BASE}/wild_persistence_results.json', 'w'), indent=2)

To test generalization beyond staged data, the CNN-LSTM trained on GMDCSA-24 (6 pose features, threshold 0.40) was evaluated without retraining on 10 real-world fall clips collected from social media (9 accidental falls in malls, stores and workplaces, and 1 staged clip). Using a 6 s sliding window, the model detected 8 of 9 real falls (recall 0.89) with a mean alert latency of 4.0 s, compared with 0 of 9 for a posture-and-timer rule baseline. Its main weakness was false alarms: 6 alerts occurred outside fall periods, half of them triggered by bystanders crouching or kneeling beside the victim. This mirrors the false alarms observed on GMDCSA-24 (lying down, push-ups) and indicates that the model responds to low body posture rather than to the fall transition itself. Requiring the probability to stay above threshold for consecutive windows reduced false alarms from 6 to 1 but also lowered recall to 0.56, partly because several clips end shortly after the fall. The remaining misses involved small, distant victims or crowded scenes in which the single-person pose estimator likely tracked a bystander, which motivates a multi-person pose and tracking base. Ground-truth timings were annotated at 0.5 s resolution, and the clips were used for evaluation only.

In [ ]:
from google.colab import files
base = '/content/drive/MyDrive/J26-DS-320/datasets/'
for f in ['wild_test_results.json', 'wild_persistence_results.json', 'wild_test_timeline.png']:
    files.download(base + f)